# CTMatch — Tier 2a: Zero-Shot LLM Reranker

**Goal:** Score (patient, trial) pairs with log P(yes) − log P(no) from a small open LLM,
then measure NDCG@10 on the TREC22 holdout and all-184 topics.

**Three evals (in order):**
1. **Gate — clf-v4 reproduction:** Standalone scorer must match 0.7460 all-184 / 0.6388 TREC22 before LLM work counts.
2. **LLM standalone:** LLM scores all judged docs per topic → NDCG@10.
3. **clf-v4 → LLM pipeline:** clf ranks all judged docs, LLM reranks top-K → NDCG@10.

**Model options (set `LLM_MODEL` in config):**
- `Qwen/Qwen2.5-7B-Instruct` — Apache-2.0, no HF gating required (default)
- `mistralai/Mistral-7B-Instruct-v0.3` — requires accepted license on HF
- `meta-llama/Meta-Llama-3-8B-Instruct` — requires accepted license on HF

---

**Results (TREC22, 50 topics, Qwen2.5-7B-Instruct):**

| System | NDCG@10 | MRR |
|---|---|---|
| clf-v4 (BioLinkBERT-large) | 0.6383 | 0.7477 |
| Qwen2.5-7B standalone (zero-shot) | 0.6269 | 0.7720 |
| **clf-v4 → Qwen2.5-7B (top-50)** | **0.6485** | **0.7759** |

Pipeline delta vs clf-v4: NDCG@10 +0.010 (+1.6%), MRR +0.028 (+3.8%).

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
                        capture_output=True, text=True)
print(result.stdout.strip() or 'No GPU found')

In [ ]:
!pip install -q transformers datasets huggingface_hub tqdm bitsandbytes accelerate

In [ ]:
import os
os.environ['HF_TOKEN'] = ''  # paste your READ token here

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ======== CONFIG ========
DATA_ROOT      = '/content/drive/MyDrive/ct_data23'
QRELS_FILE     = f'{DATA_ROOT}/unified_qrels.jsonl'
RESULTS_FILE   = f'{DATA_ROOT}/llm_reranker_results.json'

CLF_CHECKPOINT = 'semaj83/ctmatch-clf-v4'
CLF_GATE_ALL   = 0.7460   # expected NDCG@10 all-184
CLF_GATE_T22   = 0.6388   # expected NDCG@10 TREC22

# LLM to use as reranker (pick one):
LLM_MODEL = 'Qwen/Qwen2.5-7B-Instruct'             # Apache-2.0, no gating
# LLM_MODEL = 'mistralai/Mistral-7B-Instruct-v0.3'  # requires accepted license
# LLM_MODEL = 'meta-llama/Meta-Llama-3-8B-Instruct' # requires accepted license

CLF_BATCH_SIZE = 32   # for BioLinkBERT-large cross-encoder scoring
LLM_BATCH_SIZE = 4    # for LLM forward pass (reduce to 1 if OOM)
LLM_MAX_LEN    = 2048 # tokenizer max_length; Qwen/Mistral/Llama all support >= 4096

TOP_K_RERANK   = 50   # clf selects top-K; LLM reranks those K for the pipeline eval

# Toggle: True = TREC22 only (50 topics, fast gate check); False = all 184
TREC22_ONLY = True

## Load judged pool and doc texts

In [ ]:
import json
from collections import defaultdict

# Load unified_qrels.jsonl: {source, topic_id, topic_text, doc_id, label}
records = []
with open(QRELS_FILE) as f:
    for line in f:
        records.append(json.loads(line))

# Build per-topic structures
topic2text = {}          # topic_id -> topic_text
topic2source = {}        # topic_id -> source (trec21 / trec22 / kz)
topic2rel = defaultdict(dict)  # topic_id -> {doc_id: int(label)}

for r in records:
    tid = r['topic_id']
    topic2text[tid] = r['topic_text']
    topic2source[tid] = r['source']
    topic2rel[tid][r['doc_id']] = int(r['label'])

all_topics = list(topic2text.keys())
trec22_topics = [t for t in all_topics if topic2source[t] == 'trec22']
eval_topics = trec22_topics if TREC22_ONLY else all_topics

total_pairs = sum(len(topic2rel[t]) for t in eval_topics)
print(f'Eval topics: {len(eval_topics)} ({"TREC22 only" if TREC22_ONLY else "all 184"})')
print(f'Total judged pairs in eval set: {total_pairs:,}')
print(f'Avg judged per topic: {total_pairs / len(eval_topics):.0f}')

In [ ]:
from datasets import load_dataset

print('Loading doc texts from semaj83/ctmatch_ir ...')
ids_ds   = load_dataset('semaj83/ctmatch_ir', data_files='index2docid.txt', split='train')
texts_ds = load_dataset('semaj83/ctmatch_ir', data_files='doc_texts.txt',   split='train')
docid2text = dict(zip(ids_ds['text'], texts_ds['text']))
print(f'Loaded {len(docid2text):,} doc texts')

# Verify coverage for eval set
missing = sum(1 for t in eval_topics for d in topic2rel[t] if d not in docid2text)
print(f'Missing doc texts in eval set: {missing}')

## Metric functions (matching ctmatch eval_utils.py exactly)

In [ ]:
import numpy as np

def calc_ndcg(ranked_ids, doc2rel, k=10):
    """Graded NDCG@k — identical to ctmatch/evaluation/eval_utils.py."""
    dcg = sum(
        (2 ** doc2rel.get(doc_id, 0) - 1) / np.log2(i + 2)
        for i, doc_id in enumerate(ranked_ids[:k])
    )
    ideal_rels = sorted(doc2rel.values(), reverse=True)[:k]
    idcg = sum((2 ** rel - 1) / np.log2(i + 2) for i, rel in enumerate(ideal_rels))
    return dcg / idcg if idcg > 0 else 0.0

def calc_mrr(ranked_ids, doc2rel, pos_val=2):
    for i, doc_id in enumerate(ranked_ids):
        if doc2rel.get(doc_id, 0) == pos_val:
            return 1.0 / (i + 1)
    return 0.0

def eval_rankings(topic2ranked_ids, k=10):
    ndcgs, mrrs = [], []
    for tid, ranked_ids in topic2ranked_ids.items():
        ndcgs.append(calc_ndcg(ranked_ids, topic2rel[tid], k=k))
        mrrs.append(calc_mrr(ranked_ids, topic2rel[tid]))
    return {'ndcg@10': np.mean(ndcgs), 'mrr': np.mean(mrrs), 'n_topics': len(ndcgs)}

print('Metric functions ready.')

## Stage 1: clf-v4 scoring (reproduction gate)

Score all judged docs per topic with BioLinkBERT-large clf-v4 using softmax P(relevant).  
This must reproduce NDCG@10 ≥ 0.7460 (all-184) / 0.6388 (TREC22) before the LLM stage runs.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

print(f'Loading clf-v4: {CLF_CHECKPOINT}')
clf_tokenizer = AutoTokenizer.from_pretrained(CLF_CHECKPOINT)
clf_model = (
    AutoModelForSequenceClassification
    .from_pretrained(CLF_CHECKPOINT)
    .to(device)
    .eval()
)

# Determine relevant column from model config (don't hardcode 2)
id2label = clf_model.config.id2label
relevant_col = [int(k) for k, v in id2label.items() if v == 'relevant']
assert len(relevant_col) == 1, f'Expected one relevant col, got id2label={id2label}'
relevant_col = relevant_col[0]
print(f'id2label: {id2label}')
print(f'Ranking by column {relevant_col} (P(relevant))')

In [ ]:
from tqdm.auto import tqdm

def clf_score_batch(topic_text, doc_texts, batch_size=CLF_BATCH_SIZE):
    """Return softmax P(relevant) for each doc — matches pipeline.classifier_filter."""
    all_probs = []
    for i in range(0, len(doc_texts), batch_size):
        chunk = doc_texts[i:i + batch_size]
        inputs = clf_tokenizer(
            [topic_text] * len(chunk), chunk,
            return_tensors='pt', padding=True,
            truncation='longest_first', max_length=512
        ).to(device)
        with torch.no_grad():
            logits = clf_model(**inputs).logits
        probs = F.softmax(logits, dim=1)[:, relevant_col].cpu().numpy()
        all_probs.extend(probs.tolist())
    return all_probs

# Score all judged docs per topic
clf_topic2scores = {}   # topic_id -> {doc_id: float}
clf_topic2ranked = {}   # topic_id -> [doc_id sorted by clf score desc]

for tid in tqdm(eval_topics, desc='clf-v4 scoring'):
    doc_ids = list(topic2rel[tid].keys())
    doc_texts = [docid2text.get(d, '') for d in doc_ids]
    topic_text = topic2text[tid]

    scores = clf_score_batch(topic_text, doc_texts)
    scored = sorted(zip(doc_ids, scores), key=lambda x: -x[1])
    clf_topic2scores[tid] = {d: s for d, s in scored}
    clf_topic2ranked[tid] = [d for d, _ in scored]

print(f'Scored {len(clf_topic2ranked)} topics.')

In [ ]:
# Reproduction gate — top-10 ranking, matching evaluator behavior
clf_top10 = {tid: ranked[:10] for tid, ranked in clf_topic2ranked.items()}
clf_metrics = eval_rankings(clf_top10)

print('=== clf-v4 standalone (reproduction gate) ===')
print(f'  NDCG@10: {clf_metrics["ndcg@10"]:.4f}')
print(f'  MRR:     {clf_metrics["mrr"]:.4f}')
print(f'  Topics:  {clf_metrics["n_topics"]}')

expected = CLF_GATE_T22 if TREC22_ONLY else CLF_GATE_ALL
tol = 0.005  # allow ±0.5 pp rounding
gap = clf_metrics['ndcg@10'] - expected
status = 'PASS' if abs(gap) <= tol else 'FAIL — STOP AND INVESTIGATE'
print(f'\nGate ({"TREC22" if TREC22_ONLY else "all-184"}): expected {expected:.4f}, got {clf_metrics["ndcg@10"]:.4f} → {status}')
if abs(gap) > tol:
    raise RuntimeError(f'Gate FAILED (gap={gap:+.4f}). Fix before running LLM stage.')

In [ ]:
# Free GPU memory before loading LLM
import gc
del clf_model
gc.collect()
torch.cuda.empty_cache()
print('clf-v4 unloaded. GPU memory freed.')

## Stage 2: LLM scoring

Score each (topic, doc) pair by computing log P(yes) − log P(no) at the first generated token.  
No generation — single forward pass per pair, checked against answer token IDs.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f'Loading LLM: {LLM_MODEL}')
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

# Left-pad for batched inference (decoder-only LLMs)
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token
llm_tokenizer.padding_side = 'left'

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    torch_dtype=torch.float16,
    device_map='auto',
)
llm_model.eval()

# Resolve first-layer device (safe with device_map='auto')
llm_device = next(llm_model.parameters()).device
print(f'LLM loaded on {llm_device}')

In [ ]:
# Locate yes/no answer token IDs for this tokenizer
def get_answer_ids(tokenizer):
    yes_ids, no_ids = [], []
    for v in ['yes', 'Yes', ' yes', ' Yes']:
        toks = tokenizer.encode(v, add_special_tokens=False)
        if len(toks) == 1:
            yes_ids.append(toks[0])
    for v in ['no', 'No', ' no', ' No']:
        toks = tokenizer.encode(v, add_special_tokens=False)
        if len(toks) == 1:
            no_ids.append(toks[0])
    return list(set(yes_ids)), list(set(no_ids))

yes_ids, no_ids = get_answer_ids(llm_tokenizer)
assert yes_ids, 'No single-token yes variants found — check tokenizer'
assert no_ids,  'No single-token no  variants found — check tokenizer'
print(f'yes token IDs: {yes_ids} → {[llm_tokenizer.decode([i]) for i in yes_ids]}')
print(f'no  token IDs: {no_ids}  → {[llm_tokenizer.decode([i]) for i in no_ids]}')

In [ ]:
def make_prompt(topic_text: str, doc_text: str) -> str:
    messages = [{
        'role': 'user',
        'content': (
            'You are a clinical trial matching expert.\n\n'
            f'Patient:\n{topic_text}\n\n'
            f'Trial eligibility criteria:\n{doc_text}\n\n'
            'Is this patient likely eligible for this trial? '
            'Answer with a single word: yes or no.'
        )
    }]
    return llm_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

print('Prompt template ready.')
print('--- example prompt (first 400 chars) ---')
ex = list(eval_topics)[0]
ex_doc = list(topic2rel[ex].keys())[0]
print(make_prompt(topic2text[ex], docid2text.get(ex_doc, ''))[:400])

In [ ]:
# Sanity check: generate 1 token on 5 examples and confirm first output is yes/no
print('=== LLM sanity check (first generated token) ===')
sample_topics = list(eval_topics)[:5]
for tid in sample_topics:
    doc_id = list(topic2rel[tid].keys())[0]
    prompt = make_prompt(topic2text[tid], docid2text.get(doc_id, ''))
    inputs = llm_tokenizer(prompt, return_tensors='pt', truncation=True,
                           max_length=LLM_MAX_LEN).to(llm_device)
    with torch.no_grad():
        out = llm_model.generate(**inputs, max_new_tokens=3, do_sample=False)
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    generated = llm_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    rel_label = topic2rel[tid][doc_id]
    print(f'  topic={tid} doc={doc_id} rel={rel_label} → "{generated}"')

print('\nIf first word is not yes/no for most examples, adjust the prompt before proceeding.')

In [ ]:
def llm_score_batch(prompts, batch_size=LLM_BATCH_SIZE):
    """Return log P(yes) - log P(no) for each prompt (batched, left-padded).

    With left padding, logits[:, -1, :] is the prediction position for all
    sequences regardless of their individual lengths.
    """
    all_scores = []
    for i in range(0, len(prompts), batch_size):
        chunk = prompts[i:i + batch_size]
        inputs = llm_tokenizer(
            chunk, return_tensors='pt', padding=True,
            truncation=True, max_length=LLM_MAX_LEN
        ).to(llm_device)
        with torch.no_grad():
            logits = llm_model(**inputs).logits[:, -1, :]
        log_probs = F.log_softmax(logits, dim=-1)
        # yes_ids / no_ids are Python lists — indexing stays on whichever device log_probs is on
        yes_lp = torch.logsumexp(log_probs[:, yes_ids], dim=1)
        no_lp  = torch.logsumexp(log_probs[:, no_ids],  dim=1)
        all_scores.extend((yes_lp - no_lp).cpu().tolist())
    return all_scores

print('LLM scoring function ready.')

## Eval 2: LLM standalone

Score all judged docs for each topic with the LLM. No clf involvement. Measures raw zero-shot eligibility discrimination.

In [ ]:
llm_topic2ranked = {}  # topic_id -> [doc_id sorted by LLM score desc]

for tid in tqdm(eval_topics, desc='LLM standalone scoring'):
    doc_ids   = list(topic2rel[tid].keys())
    doc_texts = [docid2text.get(d, '') for d in doc_ids]
    topic_text = topic2text[tid]

    prompts = [make_prompt(topic_text, dt) for dt in doc_texts]
    scores  = llm_score_batch(prompts)

    scored = sorted(zip(doc_ids, scores), key=lambda x: -x[1])
    llm_topic2ranked[tid] = [d for d, _ in scored]

llm_standalone_metrics = eval_rankings({t: r[:10] for t, r in llm_topic2ranked.items()})
print('\n=== LLM standalone ===')
print(f'  NDCG@10: {llm_standalone_metrics["ndcg@10"]:.4f}')
print(f'  MRR:     {llm_standalone_metrics["mrr"]:.4f}')
print(f'  Topics:  {llm_standalone_metrics["n_topics"]}')

## Eval 3: clf-v4 → LLM pipeline

clf ranks all judged docs; LLM reranks the top-K by clf score. Final ranking: LLM-reranked top-K + clf-ordered remainder.

In [ ]:
pipeline_topic2ranked = {}

for tid in tqdm(eval_topics, desc=f'clf→LLM pipeline (top-{TOP_K_RERANK})'):
    clf_ranked = clf_topic2ranked[tid]  # all judged docs sorted by clf score
    top_k_ids  = clf_ranked[:TOP_K_RERANK]
    rest_ids   = clf_ranked[TOP_K_RERANK:]

    topic_text = topic2text[tid]
    prompts    = [make_prompt(topic_text, docid2text.get(d, '')) for d in top_k_ids]
    scores     = llm_score_batch(prompts)

    llm_reranked = [d for d, _ in sorted(zip(top_k_ids, scores), key=lambda x: -x[1])]
    pipeline_topic2ranked[tid] = llm_reranked + rest_ids  # LLM top-K then clf remainder

pipeline_metrics = eval_rankings({t: r[:10] for t, r in pipeline_topic2ranked.items()})
print(f'\n=== clf-v4 → LLM pipeline (top-{TOP_K_RERANK} reranked) ===')
print(f'  NDCG@10: {pipeline_metrics["ndcg@10"]:.4f}')
print(f'  MRR:     {pipeline_metrics["mrr"]:.4f}')
print(f'  Topics:  {pipeline_metrics["n_topics"]}')

In [ ]:
import pandas as pd

scope = 'TREC22' if TREC22_ONLY else 'all-184'
rows = [
    {'System': f'clf-v4 (BioLinkBERT-large)',   'NDCG@10': clf_metrics['ndcg@10'],              'MRR': clf_metrics['mrr'],              'n_topics': clf_metrics['n_topics']},
    {'System': f'LLM standalone ({LLM_MODEL.split("/")[-1]})', 'NDCG@10': llm_standalone_metrics['ndcg@10'], 'MRR': llm_standalone_metrics['mrr'], 'n_topics': llm_standalone_metrics['n_topics']},
    {'System': f'clf-v4 → LLM (top-{TOP_K_RERANK})',         'NDCG@10': pipeline_metrics['ndcg@10'],       'MRR': pipeline_metrics['mrr'],       'n_topics': pipeline_metrics['n_topics']},
]
df = pd.DataFrame(rows).set_index('System')
df['NDCG@10'] = df['NDCG@10'].map('{:.4f}'.format)
df['MRR']     = df['MRR'].map('{:.4f}'.format)
print(f'=== Tier 2a Results ({scope}) ===')
display(df)

delta_ndcg = float(pipeline_metrics['ndcg@10']) - float(clf_metrics['ndcg@10'])
sign = '+' if delta_ndcg >= 0 else ''
print(f'Pipeline delta vs clf-v4: NDCG@10 {sign}{delta_ndcg:.4f}')

In [ ]:
import json

results = {
    'llm_model':    LLM_MODEL,
    'scope':        'trec22' if TREC22_ONLY else 'all184',
    'top_k_rerank': TOP_K_RERANK,
    'clf_v4': {
        'ndcg@10':   clf_metrics['ndcg@10'],
        'mrr':       clf_metrics['mrr'],
        'n_topics':  clf_metrics['n_topics'],
    },
    'llm_standalone': {
        'ndcg@10':   llm_standalone_metrics['ndcg@10'],
        'mrr':       llm_standalone_metrics['mrr'],
        'n_topics':  llm_standalone_metrics['n_topics'],
    },
    f'clf_llm_top{TOP_K_RERANK}': {
        'ndcg@10':   pipeline_metrics['ndcg@10'],
        'mrr':       pipeline_metrics['mrr'],
        'n_topics':  pipeline_metrics['n_topics'],
    },
}

with open(RESULTS_FILE, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved to {RESULTS_FILE}')
print(json.dumps(results, indent=2))